In [5]:
from pymongo import MongoClient
from dotenv import dotenv_values

config = dotenv_values(".env")

def get_database():
    client = MongoClient(config["MONGO_DB_URL"])
    return client['nikodem']

database = get_database()

In [6]:
database["us"].delete_many({})
database["zus"].delete_many({})

DeleteResult({'n': 0, 'ok': 1.0}, acknowledged=True)

In [7]:
import csv

us_transfers = []

with open('us.csv', 'r') as file:
    reader = csv.reader(file)
    next(reader)
    for row in reader:        
        transferred_at = row[2].split('.')
                    
        us_transfers.append({
            "year": row[0],
            "month": row[1],
            "transferred_at": f'{transferred_at[2]}-{transferred_at[1]}-{transferred_at[0]}',
            "amount": float(row[3].replace(',', '')),
            "currency": "PLN"
        })

In [8]:
import csv

zus_transfers = []

with open('zus.csv', 'r') as file:
    reader = csv.reader(file)
    next(reader)
    for row in reader:        
        transferred_at = row[2].split('.')
                    
        zus_transfers.append({
            "year": row[0],
            "month": row[1],
            "transferred_at": f'{transferred_at[2]}-{transferred_at[1]}-{transferred_at[0]}',
            "amount": float(row[3].replace(',', '')),
            "currency": "PLN"
        })

In [9]:
from dateutil import parser
from datetime import datetime

def insert_transfer(transfer, collection_name = 'us'):
    date_of_transfer = transfer['transferred_at']
        
    collection = database[collection_name]

    us_transfer = {
        "month": int(transfer['month']),
        "year": int(transfer['year']),
        "amount": transfer['amount'],
        "transferred_at": parser.parse(date_of_transfer),
        "created_at":  parser.parse(datetime.now().isoformat())
    }

    collection.insert_one(us_transfer)
    

for us_transfer in us_transfers:
    insert_transfer(us_transfer, collection_name='us')
    
for zus_transfer in zus_transfers:
    insert_transfer(zus_transfer, collection_name='zus')